# **Lab 3: Accelerating ExecuTorch on Arm Ethos-U Neural Processors**
### NPU Inference on Corstone-320 Fixed Virtual Platform (Ethos-U85), TOSA, and Google's Model Explorer

## **Introduction**

Welcome to the **Accelerating ExecuTorch on Arm Neural Processors** lab. In this hands-on session, you will progress from considering Arm-powered CPUs, to deploying on the Ethos-U Neural Processing Unit backend. You will also learn about the Tensor Operator Set Architecture (TOSA) intermediate representation (IR), and how you can utilize tools such as Google's Model Explorer, via adapters developed by Arm, to analyze models in different formats when deploying to different backends.

You will understand why you might consider using an Ethos-U NPU over a CPU, and how you delegate to this backend while still performing lowering and quantization as before. The lab will showcase a streamlined way to simulate hardware using Arm's Fixed Virtual Platforms (FVP), including pre-existing workflow scripts and launchpads you can utilize to speed up prototyping.

**Requirements**: To complete this lab, you are recommended to utilize a machine with Linux OS (aarch64 or x86). You can also use an Arm-based Mac, but this will require additional set-up.

### **Lab Objectives:** 

The objectives of this lab are as follows:

1. **Understand Neural Processing Units (NPU) and ExecuTorch on the Ethos-U**:
   - Learn what a Neural Processing Unit is. Understand the Arm Ethos-U and the ways in which it can be configured, and identify when to choose an NPU over a CPU for Edge AI.
   - Quantize and lower PyTorch models to the Ethos-U and understand what affects delegation across the CPU and NPU, as well as the use of the Vela compiler.

2. **Understand the Tensor Operator Set Architecture (TOSA)**:
   - Learn about TOSA and understand its importance and usefulness.
   - Utilize Google's Model Explorer to investigate models in the `.tosa` representation, as well as `.pte` format.

3. **Understand the workflow for deploying models on an Ethos-U FVP**:
   - Perform deployment to Ethos-U FVP from `.pte` format, as well as starting with an initial PyTorch model.
   - Understand NPU utilization comparisons between different model types.
   - Be equipped to adapt these workflows to support different Ethos-U versions and configurations.

### **Prerequisites**

To follow this lab, you should have a basic understanding of:

- **Python programming**: Experience writing and running Python scripts.
- **AI/ML principles**: Basic understanding of AI/ML and model types.

> Note: It is assumed that the same Jupyter Kernel is used throughout this lab and that all cells are run sequentially.

> Note: Some scripts in this notebook produce long output logs. Configuring the 'Customizing Notebook Layout' settings to enable 'Output:scrolling' and setting 'Output:Text Line Limit' makes this more manageable

### **Recommended**
- It is recommended to complete Labs 1 and 2 prior to this lab.
- Non-essential but recommended prior to this lab is to complete the [Optimizing GenAI on Arm](https://www.edx.org/learn/computer-science/arm-education-ai-on-arm) course on edX or Coursera.

## **Pre-lab Set-up**

Ensure you have followed all the steps outlined in `Lab_0_Setup.md` and are using the correct virtual python environment.

To keep the notebook output clean and focus on learning, we will suppress some verbose, non-critical warnings.

In [ ]:
import warnings
import logging
import os

# Suppress Python warnings (including FutureWarning)
warnings.filterwarnings("ignore")

# Reduce logging verbosity
logging.getLogger().setLevel(logging.ERROR)

# Suppress Python warning environment noise
os.environ["PYTHONWARNINGS"] = "ignore"

## **What are Neural Processing Units? What is the Ethos-U?**

Neural Processing Units (NPUs) are specialized hardware accelerators designed to efficiently execute machine learning workloads. Unlike general-purpose CPUs, which are optimized for flexibility, NPUs are purpose-built for the mathematical operations that dominate neural networks, especially multiply-accumulate (MAC) operations used in convolutions and matrix multiplications. By focusing on these core operations and using optimized dataflows, memory access patterns, and quantized arithmetic (often INT8), NPUs can deliver significantly higher performance per watt. This makes them especially valuable in edge devices where power, thermal limits, and real-time responsiveness are critical.

The Arm Ethos-U is a family of NPUs designed specifically for embedded and Edge AI. The Ethos-U IP, which includes U55, U65, and U85, scales across a range of performance points through configurable numbers of MAC units (up to 2048 on the U85), enabling designers to balance throughput, area, and power. Ethos-U NPUs are often paired with Cortex-M processors in deeply embedded systems, and in higher-performance edge AI systems can be integrated alongside Cortex-A application processors. The Ethos-U handles the neural network inference workload, while the Cortex CPU manages control flow, preprocessing, postprocessing, and system integration. 

The Ethos-U supports both ExecuTorch and TFLite, and uses the Arm Vela compiler to optimize and translate models into an efficient command stream, performing operator partitioning, memory planning, and hardware-specific optimization. In the ExecuTorch flow, the model is first lowered to the Arm Ethos-U backend, where supported subgraphs are converted into a TOSA flatbuffer representation. Vela then consumes this serialized TOSA graph (rather than native ExecuTorch) and compiles it into the Ethos-U command stream. We will discuss more about TOSA later in the lab. Ethos-U supports quantized INT8 and INT16 data representation and does not support float representation.

A good summary of CPU vs NPU is: use Ethos-U when you have a quantized model with good operator coverage and need efficient, scalable Edge AI acceleration; use the CPU when flexibility, simplicity, or unsupported operators matter more than maximum efficiency. If your workload is dominated by neural network inference and you care about performance per watt, latency, or freeing the CPU for other tasks, Ethos-U is a suitable choice.

| Feature                     | Ethos-U55                         | Ethos-U65                          | Ethos-U85                          |
|-----------------------------|-----------------------------------|-------------------------------------|-------------------------------------|
| Target Systems              | Cortex-M microcontrollers         | Cortex-A class edge SoCs           | High-performance edge AI SoCs      |
| MAC Units (configurable)    | 32 – 256                          | Up to 512                          | Up to 2048                         |
| Performance Scaling         | Linear with MAC count             | Higher bandwidth & scaling         | Major uplift for larger models and full transformer support     |
| Memory Interface            | Optimized for SRAM                | Wider memory interface             | Larger memory & bandwidth support  |
| Typical Use Cases           | Keyword spotting, vision MCU AI   | Smart cameras, edge gateways       | Advanced vision, multimodal, LLM-edge inference |

## **Ethos-U ExecuTorch Workflow**

As in Lab 2, we will primarily use MobileNetV2. Starting with the PyTorch model, we will step through how to lower to the Ethos-U, which is very similar to how we previously lowered to the XNNPACK backend. 

First we define our `CompileSpec`. This compile specification configures the Arm Vela compiler to target an Ethos-U85 with 256 MAC units (`ethos-u85-256`) and assumes a specific system configuration using mid-range DRAM bandwidth (`Ethos_U85_SYS_DRAM_Mid`). The `Shared_Sram` memory mode indicates that the NPU shares SRAM with the CPU rather than using dedicated tightly-coupled memory, and `--output-format=raw` instructs Vela to generate the hardware-ready Ethos-U command stream without wrapping it in a framework container, so it can be directly embedded and invoked by the target runtime or firmware.

Next we create several reusable helper functions to perform the lowering without guard checks (to ensure more stable and consistent lowering), and allow us to assess the delegation after the lowering. The key difference compared with XNNPACK is that now we use the `EthosUPartioner`. Behind the scenes `to_edge_transform_and_lower` is now performing different steps: lowering to a core ATen operator set, partioning subgraphs that can run on the Ethos-U, lowering these subgraphs to TOSA and serializing them, before using Vela to compile into the command stream. For a simple reference workflow covering these steps with a simplified neural network performing addition on the Ethos-U55, see the [ethos_u_minimal_example](https://github.com/pytorch/executorch/blob/main/examples/arm/ethos_u_minimal_example.ipynb), which you have already obtained through cloning ExecuTorch. You are also encouraged to use this lab alongside the [official documentation](https://docs.pytorch.org/executorch/0.3/executorch-arm-delegate-tutorial.html).

In [ ]:
import torch
from torchvision.models import mobilenet_v2, MobileNet_V2_Weights
from executorch.backends.arm.ethosu import EthosUCompileSpec, EthosUPartitioner
from executorch.backends.arm.quantizer import EthosUQuantizer, get_symmetric_quantization_config
from torchao.quantization.pt2e.quantize_pt2e import prepare_pt2e, convert_pt2e
from executorch.exir import EdgeCompileConfig, ExecutorchBackendConfig, to_edge_transform_and_lower

# -------------------------
# Config
# -------------------------
SAVE_DIR = os.environ.get("SAVE_DIR", "NPU_Lab_ExecuTorch/out_pte")
os.makedirs(SAVE_DIR, exist_ok=True)

example_inputs = (torch.randn(1, 3, 224, 224),)

compile_spec = EthosUCompileSpec(
    target="ethos-u85-256",
    system_config="Ethos_U85_SYS_DRAM_Mid",
    memory_mode="Shared_Sram",
    extra_flags=["--output-format=raw"],
)

# -------------------------
# Helper Functions
# -------------------------
def export_ep_no_guards(model: torch.nn.Module, example_inputs):
    model = model.eval()
    ep = torch.export.export(model, example_inputs)
    gm = ep.module(check_guards=False)               
    return torch.export.export(gm, example_inputs)   

def lower_to_ethosu_and_save(ep, compile_spec, pte_path: str):
    edge_pm = to_edge_transform_and_lower(
        ep,
        compile_config=EdgeCompileConfig(_check_ir_validity=False),
        partitioner=[EthosUPartitioner(compile_spec)],
    )
    et_pm = edge_pm.to_executorch(ExecutorchBackendConfig(extract_delegate_segments=True))
    et_pm.save(pte_path)
    return edge_pm

def _get_graph_module_from_edge_pm(edge_pm):
    if hasattr(edge_pm, "exported_program"):
        ep = edge_pm.exported_program()
        if hasattr(ep, "graph_module"):
            return ep.graph_module
    for attr in ["graph_module", "gm", "_graph_module"]:
        if hasattr(edge_pm, attr):
            return getattr(edge_pm, attr)
    raise RuntimeError("Couldn't locate a graph_module on edge_pm (API mismatch).")

def delegation_stats(edge_pm, name):
    gm = _get_graph_module_from_edge_pm(edge_pm)
    nodes = list(gm.graph.nodes)

    delegate_nodes = [
        n for n in nodes
        if n.op == "call_function"
        and ("call_delegate" in str(n.target)
             or "executorch_call_delegate" in str(n.target))
    ]

    print(f"\n{name}")
    print(f"  Top-level FX nodes: {len(nodes)}")
    print(f"  Delegate call nodes: {len(delegate_nodes)}")

    if len(delegate_nodes) == 0:
        print("  ➜ No Ethos-U delegation detected (graph runs on host).")
    else:
        print("  ➜ Ethos-U delegate present.")
        print("    The heavy compute ops (convs, etc.) are typically encapsulated inside this delegate segment.")

To begin, we will utilize the standard FP32 MobileNetV2 model and lower to Ethos-U without performing quantization. Consider what you might expect to happen in this case?

In [ ]:
model = mobilenet_v2(weights=MobileNet_V2_Weights.DEFAULT).eval()
ep_float = export_ep_no_guards(model, example_inputs)

pte_float = os.path.join(SAVE_DIR, "mobilenetv2_fp32_ethosu.pte")
edge_pm_float = lower_to_ethosu_and_save(ep_float, compile_spec, pte_float)
print("Saved:", pte_float)

delegation_stats(edge_pm_float, "MobileNetV2 (FP32)")

If you guessed that we would be unable to delegate to the Ethos-U, you are correct. We mentioned earlier that the Ethos-U does not support floating-point representation. Instead we must quantize to INT8/INT16 format.

Our `delegation_stats` helper function has reported no delegated subgraphs, so the model will execute entirely on the CPU. Since the Ethos-U is always paired with a Cortex-M or A, in this situation the CPU provides a fallback option. We also received no output from Vela, as it was not utilized as no compilation was needed.

Now we will quantize to INT8 with the `EthosUQuantizer`. As with XNNPACK, backends have their own backend-specific quantizer.

In [ ]:
model = mobilenet_v2(weights=MobileNet_V2_Weights.DEFAULT).eval()

# Export
ep = torch.export.export(model, example_inputs)
gm = ep.module(check_guards=False)

# Quantizer (Ethos-U) + symmetric configuration
quantizer = EthosUQuantizer(compile_spec)
quantizer.set_global(get_symmetric_quantization_config())

# Post-training quantization
q_gm = prepare_pt2e(gm, quantizer)
with torch.no_grad():
    q_gm(*example_inputs)  # Calibration pass
q_gm = convert_pt2e(q_gm)

# Re-export quantized graph module for lowering
ep_int8 = torch.export.export(q_gm, example_inputs)

# Lower to Ethos-U
pte_int8 = os.path.join(SAVE_DIR, "mobilenetv2_int8_ethosu.pte")
edge_pm_int8 = lower_to_ethosu_and_save(ep_int8, compile_spec, pte_int8)
print("Saved:", pte_int8)

This Vela report shows that the quantized INT8 MobileNetV2 has been fully compiled for an Ethos-U85 with 256 MAC units, using a mid-range DRAM system configuration and shared SRAM memory model - exactly as the `CompileSpec` defined. Importantly, since CPU operators = 0 and NPU operators = 66 (100%), the entire network was successfully delegated to the NPU with no CPU fallback. This does not mean that the CPU does no work. We will see shortly that the CPU remains involved for high-level orchestration.

The memory section shows how much on-chip SRAM (~1.47 MB) and external DRAM (~3.36 MB) the compiled model requires, along with estimated bandwidth usage. The greatest bandwidth happens in SRAM (fast, on-chip memory), while DRAM traffic is relatively low, and mainly used for weights, which helps improve performance and energy efficiency.

The model requires ~301 million MACs per inference, which is the total amount of neural compute work MobileNetV2 performs per input. Since this is running at a configured 1 GHz NPU clock and fully offloaded, this gives you a strong indication of the achievable throughput and efficiency on this Ethos-U85 configuration. The .pte file saved at the end contains the ExecuTorch program with the embedded Ethos-U command stream ready for deployment.

Take some time to read through the various metrics and familiarize yourself with them.

**To be clear, these figures are compiler estimates generated by Vela based on the model and target configuration — they are not direct runtime performance measurements taken from hardware.**

Now we can examine our delegation statistics:

In [ ]:
delegation_stats(edge_pm_int8, "MobileNetV2 (int8 PT2E)")

We can now see that the top-level FX nodes have dropped significantly, compared with the FP32 model that could not be delegated. As expected, we also have a call node to a delegated subgraph. The CPU handles only minimal orchestration (such as passing inputs and retrieving outputs), while the heavy compute operations, such as convolutions and other core layers, are executed inside the delegated subgraph on the Ethos-U NPU.

We will now move on to discussing TOSA in detail.

## **The Tensor Operator Set Architecture (TOSA)**

### **What is TOSA?**
The **Tensor Operator Set Architecture (TOSA)** is a minimal and stable set of tensor-level operators designed to **standardize machine learning model execution across hardware platforms**. [TOSA](https://www.mlplatform.org/tosa/) is framework-agnostic and includes precise functional and numerical definitions to ensure consistent results across CPUs, GPUs, and NPUs. The latest specification can be found [here](https://www.mlplatform.org/tosa/tosa_spec.html), and is provided as a [GitHub repository](https://github.com/arm/tosa-specification) as well.

### **Why Do We Need TOSA?**
Modern ML frameworks such as [Executorch](https://github.com/pytorch/executorch), [LiteRT](https://ai.google.dev/edge/litert), and [JAX](https://github.com/jax-ml/jax) define hundreds (sometimes thousands) of distinct operators.  This diversity creates challenges when deploying models across platforms or converting them between frameworks (forexample, PyTorch → TensorFlow conversion is currently non-trivial due to different tensor layout conventions). 

**TOSA solves this problem** by providing a common set of standard operators that act as a stable intermediate representation (IR), a kind of “middle language” between source code and machine instructions that can be further compiled to a diverse range of backend targets. You can think of it as a bit like an assembly language. Subtle variations in the way operators are implemented in different frameworks, such as `CONV2D` for [2D convolutions](https://www.mlplatform.org/tosa/tosa_spec_1_0_1.html#_conv2d) are specified in detail so that for ML developers, a model will perform as expected across devices and frameworks with an acceptably low level of rounding errors. 

### **TOSA in the ExecuTorch Compilation Flow**

TOSA was added to PyTorch and ExecuTorch through [collaboration between Arm and Meta](https://developer.arm.com/community/arm-community-blogs/b/ai-blog/posts/executorch-and-tosa-enabling-pytorch-on-arm-platforms) in 2023.

**So, why did we not use TOSA in Labs 1 and 2?**

In the earlier XNNPACK CPU labs, TOSA was not used. Instead, ExecuTorch lowered models directly to the XNNPACK backend, which is a highly optimized CPU kernel library. Because the target was a general-purpose CPU with an established execution provider, there was no need for an additional intermediate representation. The flow was simpler: PyTorch → XNNPACK, avoiding the extra PyTorch → TOSA → backend step.

In principle, we could have lowered to TOSA first, but there would have been little practical benefit. XNNPACK is already a mature, CPU-optimized kernel library with its own well-defined operator expectations, and ExecuTorch can lower PyTorch operators directly to it. TOSA is most valuable when creating a stable contract between frameworks and new systems, particularly those that are heterogeneous (e.g., systems that include NPUs/AI accelerators or GPUs). Instead of implementing hundreds of framework-specific operators for each new device, a vendor can implement the smaller, stable TOSA operator set and rely on ExecuTorch to lower models into that form.

In the Ethos-U compilation flow demonstrated earlier, TOSA is already being used under the hood. ExecuTorch lowers supported portions of the model into a serialized TOSA representation, which is then passed to the Arm Vela compiler. Vela compiles this TOSA graph into an optimized command stream for the Ethos-U NPU.

### **Key Benefits of TOSA**
- **Standardization**: A consistent, well-defined target for both ML frameworks and hardware vendors.  
- **Portability**: TOSA provides precise operator definitions to promote consistent numerical behavior across compliant implementations.  
- **Future-Proofing**: Future novel operators not directly in TOSA can still be expressed using existing TOSA primitives, ensuring long-term flexibility.  

If a model contains operators that are not supported in TOSA or by the Ethos-U backend, delegation may fragment into multiple smaller subgraphs, with unsupported operators running on the CPU between NPU segments. This can reduce performance due to increased orchestration and memory transfers. To demonstrate this effect, we will introduce an LRN operator into MobileNetV2.

Local Response Normalization (LRN) is an operation used in early convolutional neural networks that normalizes a neuron’s activation based on the activity of neighboring channels, encouraging competition between feature maps (as seen in models like AlexNet). LRN is not a native TOSA operator, and it will not be delegated through the TOSA-based Ethos-U compilation flow.

In [ ]:
import torch.nn as nn
import torch.nn.functional as F

class MobileNetV2_LRN(nn.Module):
    def __init__(self):
        super().__init__()
        base = mobilenet_v2(weights=MobileNet_V2_Weights.DEFAULT).eval()
        self.features = base.features
        self.classifier = base.classifier
        self.lrn = nn.LocalResponseNorm(5)

    def forward(self, x):
        x = self.features(x)
        x = self.lrn(x)                    # <- breaks TOSA compatibility
        x = F.adaptive_avg_pool2d(x, (1, 1))
        x = torch.flatten(x, 1)
        x = self.classifier(x)
        return x

# Build model
lrn_model = MobileNetV2_LRN().eval()

# Export (no guards)
ep = torch.export.export(lrn_model, example_inputs)
gm = ep.module(check_guards=False)

# Quantize (PT2E flow)
quantizer = EthosUQuantizer(compile_spec)
quantizer.set_global(get_symmetric_quantization_config())

q_gm = prepare_pt2e(gm, quantizer)
with torch.no_grad():
    q_gm(*example_inputs)
q_gm = convert_pt2e(q_gm)

ep_lrn_int8 = torch.export.export(q_gm, example_inputs)

# Lower
pte_lrn_int8 = os.path.join(SAVE_DIR, "mobilenetv2_lrn_int8_ethosu.pte")
edge_pm_lrn_int8 = lower_to_ethosu_and_save(ep_lrn_int8, compile_spec, pte_lrn_int8)

print("Saved:", pte_lrn_int8)

This time, we can see Vela has produced two subgraphs, as opposed to one. Lets look at the delegation statistics and then discuss.

In [ ]:
# Compare delegation
delegation_stats(edge_pm_lrn_int8, "MobileNetV2 + LRN (int8)")

When we inserted LRN into MobileNetV2 and re-ran lowering, the delegation statistics changed from a single delegate call to two delegate call nodes and an increase in top-level FX nodes (from 9 to 19). This tells us that delegation has fragmented. Instead of the entire network being wrapped into one large Ethos-U subgraph, the model has been split into multiple regions: supported sections are offloaded to the NPU, while unsupported parts (such as LRN) remain on the CPU in between those regions.

This fragmentation is exactly what we would expect when introducing an operator that is not natively supported in the TOSA → Vela → Ethos-U flow. The partitioner isolates the unsupported operation, preventing it from being included in the NPU segment. Whilst the heavy convolutional compute is still accelerated, the model now incurs additional orchestration and memory movement overhead between CPU and NPU.

The key learning takeaway is that operator support directly impacts delegation quality. When all operators are supported, you get a single large NPU subgraph and maximum acceleration. When unsupported operators are introduced, delegation fragments, reducing efficiency even if most of the compute is still offloaded. Understanding this relationship between operator coverage, TOSA lowering, and backend support is critical when optimizing models for edge NPUs like Ethos-U.

We will now generate TOSA intermediate representations for our three versions of MobileNetV2: FP32, INT8-quantized, and INT8 with an inserted LRN layer. For each exported model, we create a TOSA compile specification, partition, and run `to_edge_transform_and_lower` to lower supported subgraphs into serialized `.tosa` files. This allows us to compare how operator support differs across float vs. quantized models, and to observe how inserting LRN affects TOSA lowering. 

It is important to understand that in this step we are lowering the models to TOSA, not lowering them to Ethos-U hardware. TOSA is an intermediate representation that supports both floating-point and integer profiles (for example, `TOSA-1.0+FP` and `TOSA-1.0+INT`). Because TOSA includes a floating-point profile, an FP32 MobileNetV2 can be successfully lowered to a `.tosa` file. This simply means the model can be expressed using the standardized TOSA operator set. It does not mean the model can run on the Ethos-U NPU. As we demonstrated earlier, Ethos-U supports INT8/INT16, not FP representation.

In [ ]:
from pathlib import Path
from torch.export import export as torch_export
from executorch.exir import to_edge_transform_and_lower, EdgeCompileConfig
from executorch.backends.arm.tosa import TosaSpecification
from executorch.backends.arm.tosa.compile_spec import TosaCompileSpec
from executorch.backends.arm.util._factory import create_partitioner

# Helper: output TOSA files
def dump_tosa(ep, profile_str: str, dump_dir: Path, label: str):
    dump_dir.mkdir(parents=True, exist_ok=True)

    tosa_spec = TosaSpecification.create_from_string(profile_str)
    compile_spec = TosaCompileSpec(tosa_spec)
    compile_spec.dump_intermediate_artifacts_to(str(dump_dir))

    partitioner = create_partitioner(compile_spec)

    _ = to_edge_transform_and_lower(
        ep,
        partitioner=[partitioner],
        compile_config=EdgeCompileConfig(_check_ir_validity=False),
    )

    tosa_files = list(dump_dir.rglob("*.tosa"))

    print(f"\n{label}")
    print(f"  Profile: {profile_str}")
    print(f"  Dump dir: {dump_dir.resolve()}")
    if tosa_files:
        print(f"  .tosa files found: {len(tosa_files)}")
    else:
        print("  No .tosa files generated (likely nothing lowered for this profile).")

# Output TOSA files
BASE_DUMP = Path("NPU_Lab_ExecuTorch/out_tosa")

dump_tosa(ep_float,      "TOSA-1.0+FP",  BASE_DUMP / "mv2_float_fp",      "MobileNetV2 (float)")
dump_tosa(ep_int8,       "TOSA-1.0+INT", BASE_DUMP / "mv2_int8_int",      "MobileNetV2 (int8)")
dump_tosa(ep_lrn_int8,   "TOSA-1.0+INT", BASE_DUMP / "mv2_lrn_int8_int",  "MobileNetV2 + LRN (int8)")

print("\nDone. Inspect the generated .tosa files in:", BASE_DUMP.resolve())

In the LRN case, you will notice that two .tosa files were generated, corresponding to the two delegated subgraphs created after fragmentation. Each file represents a separate region of the model that was successfully lowered to TOSA. This reflects what we saw earlier in the delegation statistics - the unsupported LRN operator split the network into multiple NPU-compatible segments rather than one contiguous subgraph.

Now we are going to introduce Google's Model Explorer to visualize both `.tosa` files and `.pte` files with the TOSA and PTE adapters created by Arm.

## **Google Model Explorer**

[Google's Model Explorer](https://ai.google.dev/edge/model-explorer) is an open-source, web-based visualization tool for inspecting and analyzing machine learning model graphs. With various adapters, it allows you to load models in formats such as TOSA, PTE, TFLite, ONNX, and others, and interactively explore their structure. The tool presents the model as a hierarchical graph, where operators can be expanded, collapsed, and grouped, making it easier to understand layer structure, data flow, and backend transformations.

In the context of this lab, Model Explorer is particularly useful for visualizing how models change across compilation stages, for example, comparing an FP32 model to its INT8 version, or observing how delegation creates distinct subgraphs.

We will use pip to install Model Explorer along with adapters developed by Arm that support the `.tosa` and `.pte` formats.

In [ ]:
%pip install ai-edge-model-explorer pte-adapter-model-explorer tosa-adapter-model-explorer

Now run the bash command in your terminal to launch Model Explorer with the required adapters (make sure to activate your venv if you have not already). A full user guide for Model Explorer can be found [here](https://github.com/google-ai-edge/model-explorer/wiki/2.-User-Guide).

```bash
model-explorer --extensions=pte_adapter_model_explorer,tosa_adapter_model_explorer
```

Try loading various `.tosa` and `.pte` files and explore the differences.

Use `CTRL+C` to exit when you are done.

To activate your venv:

```bash
source ./NPU_lab_venv/bin/activate
```

Example images of the INT8 (non-LRN) `.tosa` (L) and `.pte` (R) are shown below:

![TOSA](NPU_Lab_ExecuTorch/model_explorer_imgs/tosa_example.png)
![PTE](NPU_Lab_ExecuTorch/model_explorer_imgs/pte_example.png)

Example `.tosa` and `.pte` files are also provided in this repo so that you can try out Model Explorer without having to generate your own.

When examining the PTE files, consider:
- The number of delegate nodes?
- Where the delegation boundaries occur?
- What operators occur in or out of the delegated regions?

When examining the TOSA files, consider differences in operators, such as how FP and INT have lowered to TOSA, or any structural differences.

## **Deploying to a Fixed Virtual Platform**

The next step is to deploy and run the model on a Fixed Virtual Platform (FVP) for Corstone-320.

An FVP is a fast, cycle-approximate software simulation of Arm hardware. It models processors, memory, interconnects, and peripherals, allowing you to run compiled binaries as if you were using real silicon — but entirely in software. FVPs are extremely useful during development because they let you validate software stacks, firmware, and ML deployments before physical hardware is available. They also provide deterministic, reproducible environments for experimentation and debugging. In this lab, it allows us to test end-to-end execution — CPU orchestration plus NPU acceleration — without requiring physical development boards. This bridges the gap between model compilation and real deployment, demonstrating how the compiled `.pte` artifact actually runs on a representative Arm Edge AI system.

Corstone is Arm’s reference subsystem platform for building SoCs around Cortex CPUs and optional accelerators like Ethos-U. Different Corstone variants support different IP. For example:
- Corstone-300 (Includes Cortex-M55 + Ethos-U55)
- Corstone-320 (Includes Cortex-M85 + Ethos-U85)

In this lab, we use the Corstone-320 FVP as we have been delegating to an Ethos-U85. However, it is trivial for you to change this workflow to support other Ethos variants, such as in the [minimal example](https://github.com/pytorch/executorch/blob/main/examples/arm/ethos_u_minimal_example.ipynb), where Ethos-U55 is demonstrated. 

The Arm executor runner is a small C++ “host application” that runs on the target platform. It loads and executes an ExecuTorch `.pte` program. It provides the glue between the compiled model artifact and the embedded system. It initializes the ExecuTorch runtime, loads the `.pte` from memory, sets up the required memory pools/allocators, prepares input tensors, invokes the model method (e.g., forward), and prints outputs and basic profiling/memory statistics.

### **INT8 Quantized MobileNetV2**

First we will use our non-LRN INT8 Quantized MobileNetV2 `.pte`. The script below ensures the Arm bare-metal toolchain and FVP are available, then checks that the `.pte` model file exists. It then uses CMake to build the ExecuTorch runtime libraries for an Arm bare-metal target. Next, it invokes CMake again to cross-compile the arm_executor_runner application, embedding the .pte path and specifying the Ethos-U85 configuration. Finally, it launches the Corstone-320 FVP and runs the compiled ELF binary inside the simulated system. 

In [ ]:
%%bash
set -euo pipefail

cd NPU_Lab_ExecuTorch/executorch/examples/arm

# Ensure the arm-none-eabi-gcc toolchain + FVP are on PATH.
source arm-scratch/setup_path.sh

PTE_PATH="${PTE_PATH:-$PWD/../../../out_pte/mobilenetv2_int8_ethosu.pte}"

if [ ! -f "$PTE_PATH" ]; then
  echo "ERROR: PTE not found at: $PTE_PATH"
  echo "Set PTE_PATH env var to the correct .pte location, e.g.:"
  echo "  export PTE_PATH=/full/path/to/mobilenetv2_int8_ethosu.pte"
  exit 1
fi

echo "Using PTE: $PTE_PATH"

# 1) Build ExecuTorch runtime libs for Arm baremetal
cmake --preset arm-baremetal \
  -DCMAKE_BUILD_TYPE=Release \
  -B../../cmake-out-arm ../..
cmake --build ../../cmake-out-arm --target install -j"$(nproc)"

# 2) Build executor runner for MobileNetV2 INT8 on U85
BUILD_DIR="mobilenetv2_int8_u85_runner"
cmake -DCMAKE_TOOLCHAIN_FILE="$(pwd)/ethos-u-setup/arm-none-eabi-gcc.cmake" \
  -DCMAKE_BUILD_TYPE=Release \
  -DET_PTE_FILE_PATH="$PTE_PATH" \
  -DTARGET_CPU=cortex-m85 \
  -DETHOSU_TARGET_NPU_CONFIG=ethos-u85-256 \
  -DSYSTEM_CONFIG=Ethos_U85_SYS_DRAM_Mid \
  -B"$BUILD_DIR" \
  executor_runner

cmake --build "$BUILD_DIR" -j"$(nproc)" -- arm_executor_runner

# 3) Run on the Ethos-U85 FVP
../../backends/arm/scripts/run_fvp.sh \
  --elf="$BUILD_DIR/arm_executor_runner" \
  --target=ethos-u85-256

The performance summary confirms that delegation occurred as expected. “NPU delegations: 1” indicates the model executed as a single delegated segment on the Ethos-U85, meaning the neural network compute was successfully offloaded to the NPU rather than executed on the CPU. This validates that the compiled `.pte` artifact and backend integration are functioning correctly on the target platform.

The log reports two categories of performance counters. First are the CPU-side cycle counts, including overall inference runtime and per-operator accounting. Second are the Ethos-U PMU (Performance Monitoring Unit) counters, which provide visibility into NPU activity and memory behavior. However, it is important to note the warning at the top. The Corstone FVP is not cycle accurate, so these numbers should not be interpreted as real hardware performance measurements. They are useful for functional verification and relative inspection, but not for benchmarking.

**There were a lot of stages involved in this process - can I speed this up?**

## **Simplifying the Process with the AOT Arm Compiler**

So far in this lab, we have manually walked through each stage of the deployment pipeline: exporting the PyTorch model, applying INT8 quantization, lowering through the Arm backend, compiling with Vela for Ethos-U, generating the .pte artifact, building the executor runner with CMake, and finally deploying to the Corstone-320 FVP. This step-by-step approach is valuable because it exposes what is happening at each layer of the stack — how delegation works, how TOSA fits into the flow, and how the NPU command stream is produced.

In practice, however, you would not typically perform each of these steps manually for every model. The Arm AOT (Ahead-Of-Time) compiler flow automates this entire pipeline. It takes a PyTorch model (such as MobileNetV2), applies the appropriate backend configuration, performs quantization and lowering, invokes Vela, packages the `.pte`, and can even integrate with FVP deployment scripts. In other words, it gives you a streamlined, production-oriented path from model to running system. For an example using a simplified PyTorch model, you can use this [Learning Path - Simple NN](https://learn.arm.com/learning-paths/embedded-and-microcontrollers/introduction-to-tinyml-on-arm/), and for another example using MobileNetV2, you will likely find [Learning Path - MobileNetV2](https://learn.arm.com/learning-paths/embedded-and-microcontrollers/visualizing-ethos-u-performance/) of interest.

We will now execute this flow, with PyTorch MobileNetV2. Having first understood the individual components, you are now in a strong position to appreciate what the automation is doing, and why it works.

In [ ]:
%%bash

source NPU_Lab_ExecuTorch/executorch/examples/arm/arm-scratch/setup_path.sh

source NPU_Lab_ExecuTorch/executorch/examples/arm/run.sh \
--aot_arm_compiler_flags="--delegate --quantize --intermediates mv2_u85/ --debug --evaluate" \
--output=mv2_u85 \
--target=ethos-u85-256 \
--model_name=mv2

As before, we have successfully deployed to the FVP. But this time we started with just a PyTorch model and let the script handle the intermediate quantization, lowering, and compilation steps for us. This makes the deployment flow repeatable and less error-prone.

## **End of Lab Summary – What Have We Learned?**

In this lab, we extended our exploration of ExecuTorch from CPU-based acceleration (via XNNPACK) to dedicated neural processing hardware using the Arm Ethos-U NPU. Using MobileNetV2 as our reference model, we walked step-by-step through exporting from PyTorch, applying INT8 quantization, lowering through the Arm backend, compiling with Vela, and deploying to a Corstone-320 FVP representing a Cortex-M85 + Ethos-U85 system.

We learned that successful NPU acceleration depends on more than simply enabling delegation. Operator support and quantization are critical. When the model was FP32, no delegation occurred. After INT8 quantization using the `EthosUQuantizer`, the entire network could be encapsulated into a single delegated subgraph and compiled into an Ethos-U command stream. By contrast, introducing an unsupported operator (LRN) fragmented delegation into multiple subgraphs, demonstrating how non-TOSA or unsupported operators create hard partition boundaries and reduce offload efficiency.

We also explored the role of TOSA (Tensor Operator Set Architecture) as an intermediate representation. We clarified the distinction between lowering to TOSA and delegating to Ethos-U hardware, reinforcing that TOSA expresses a model in a standardized operator form, while Ethos-U delegation requires additional backend compilation via Vela. By visualizing `.tosa` and `.pte` artifacts in Google's Model Explorer, we were able to view the differences between our versions and gain insight into how choices affect subgraph delegation.

Finally, we deployed the compiled model to a simulated Corstone-320 platform using the Arm executor runner and FVP. This demonstrated full end-to-end deployment: PyTorch → ExecuTorch → TOSA → Vela → Ethos-U → embedded execution. We also introduced the Arm AOT compiler flow, showing how the same pipeline can be automated into a reproducible production-style workflow.

In short, this lab illustrated how model structure, quantization, operator coverage, and backend configuration determine whether an NPU can be used effectively, as well as how ExecuTorch enables PyTorch models to be deployed onto deeply embedded Arm AI systems.

## **What's next?**

In addition to the various learning paths, documentation, and examples linked to earlier in this labs collection, you are encouraged to explore the [Arm Developer Edge AI](https://developer.arm.com/edge-ai) page.

You can also attempt various challenges on [Arm Developer Labs](https://arm-university.github.io/Arm-Developer-Labs/) such as:
- [Ethos-U85 for Edge Inference](https://arm-university.github.io/Arm-Developer-Labs/2025/11/27/Ethos-U85-NPU-Applications.html)
- [Always-on-AI on Ethos-U85](https://arm-university.github.io/Arm-Developer-Labs/2025/11/27/Always-On-AI-with-Ethos-U85-NPU.html)
- [Edge AI on Mobile with ExecuTorch](https://arm-university.github.io/Arm-Developer-Labs/2025/11/27/Edge-AI-On-Mobile.html)

Make sure to use the forms or arm-developer-labs@arm.com to tell us what exciting ExecuTorch applications you build on Arm-based devices - that way we can promote your work, or even potentially provide mentorship and other support.

